In [1]:
#!pip install -U langchain
#!pip install -U langchain-core
#!pip install -U langchain-community
#!pip install -U langchain-google-genai


In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferMemory
#from langchain_community.memory import ConversationBufferMemory


# 🔐 Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
)

# Simple QA tool (LLMChain)
qa_prompt = ChatPromptTemplate.from_template("Answer clearly: {question}")
qa_chain = qa_prompt | llm
qa_tool = Tool(
    name="Simple_QA",
    func=lambda question: qa_chain.invoke({"question": question}), 
    description="Answer the question without hallucination"
)

# Web search tool (Tavily)
tavily_search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Web Search", 
    handle_parsing_errors=True,
    func=tavily_search.invoke,  # Fixed: changed .run to .invoke to stop tool crash
    description="Search the internet for current and live information"
)

# 🔧 Wrap all tools in an agent
tools = [qa_tool, search_tool]
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\user\AppData\Local\Temp\ipykernel_11868\3236287020.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults
C:\Users\user\AppData\Local\Temp\ipykernel_11868\3236287020.py:32: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)
C:\Users\user\AppData\Local\Temp\ipykernel_11868\3236287020.py:42: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be rem

In [3]:
#1) Zero-Shot ReAct (simple + reliable default)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, 
    handle_parsing_errors=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")

C:\Users\user\AppData\Local\Temp\ipykernel_11868\354406309.py:3: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(
C:\Users\user\AppData\Local\Temp\ipykernel_11868\354406309.py:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_conf



> Entering new AgentExecutor chain...

Observation: Invalid Format: Missing 'Action:' after 'Thought:
Thought:

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought: I need to use the Simple_QA tool to get a summary of LangChain and its creator, then format the answer correctly.
Action: Simple_QA
Action Input: Summarize LangChain in 2 lines, then tell me who created it.

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Observation: content=[{'type': 'text', 'text': 'LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs) by chaining together various components like prompts, models, and external data sources. It bridges the gap between standalone AI models and real-world data to build advanced tools like chatbots and retrieval systems.\n\nIt was created by **Harrison Chase** in October 2022.', 'extras': {'signature': 'EmAKXgFpFH0T55Ye52j6aJuC4dUhXrPprM1d03397mMEXSB5DDUuBbqy/eAIJKkeNr3TQvpvVnwFO6gX2UOVOnTywKj2dU4dF8CCqryrag136o/uKae4ASmaZ3sRkIrYDho='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a17528-100f-7671-bc65-49853e3b96bc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 20, 'output_tokens': 78, 'total_tokens': 98, 'input_token_details': {'cache_read': 0}}
Thought:

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought: I now have the summary and the creator's name from the tool observation. I can formulate the final answer.
Final Answer: LangChain is an open-source framework designed to simplify the creation of applications using large language models by chaining together various components. It bridges the gap between standalone AI models and real-world data to build advanced tools like chatbots and retrieval systems. It was created by Harrison Chase.

> Finished chain.


In [4]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```
Thought: Do I need to use a tool? Yes
Action: Web Search
Action Input: Who created LangChain and what is it
```
Observation: [{'title': 'What is LangChain? | Oracle', 'url': 'https://www.oracle.com/database/what-is-langchain', 'content': '# What Is LangChain? A Guide\n\nJeffrey Erickson | Senior Writer | July 24, 2026\n\nLike many open source hits before it, LangChain began as a side project with great timing. In late 2022, just as developer interest in large language models was accelerating, Harrison Chase built LangChain to give himself and other developers a more systematic way to build applications around LLMs.\n\nFast-forward a few years, and LangChain is a company that positions its product as an open source framework for building agents and LLM-powered applications using reusable components and methods to connect models with enterprise data, tools, and software workflows.\n\n## What Is LangChain? [...] “Memory is becoming the defining layer of the agent stack,” said Harrison

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```
Thought: Do I need to use a tool? No
AI: LangChain is an open-source framework designed to simplify the development of applications powered by large language models by integrating them with external data and tools. It provides reusable building blocks and standardized abstractions for orchestrating complex, multi-step AI workflows. 

LangChain was created by Harrison Chase in October 2022.
```

> Finished chain.


In [5]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
    "action": "Final Answer",
    "action_input": "Here is a 3-step study path for learning LangChain:\n\n1. **Foundations & Core Concepts:** Start by understanding what Large Language Models (LLMs) are and the basic building blocks of LangChain. Learn about **Models** (interacting with various LLM providers), **Prompts** (managing prompt templates), and **Output Parsers** (formatting LLM responses).\n\n2. **Chains, Memory, and RAG:** Move on to connecting components together. Learn how to build **Chains** for sequential tasks, add **Memory** to maintain context in conversations, and implement **Retrieval-Augmented Generation (RAG)** by connecting LLMs to external documents using vector databases and retrievers.\n\n3. **Advanced Agents & Production:** Finally, explore **Agents** (allowing LLMs to decide which tools to use dynamically) and multi-agent frameworks like LangGraph. Learn how to evaluate your application, optimize prompts, and deploy your LangChain project to produ

In [6]:
#from langchain.tools import StructuredTool
from langchain_classic.agents import AgentExecutor
#from langchain.agents import initialize_agent, AgentType    ---------->  initialize_agent is no more in Lang chain
from langchain_core.tools import StructuredTool
from pydantic import BaseModel
#from langchain import hub

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)


# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(




> Entering new AgentExecutor chain...
```
{
  "action": "TitleMaker",
  "action_input": {
    "topic": "LangGraph tutorials",
    "tone": "friendly"
  }
}
```
Observation: Friendly Title: LangGraph tutorials in Practice
Thought:

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
  "action": "Final Answer",
  "action_input": "Friendly Title: LangGraph tutorials in Practice"
}
```

> Finished chain.
✅ Result: Friendly Title: LangGraph tutorials in Practice


#  5 and 6 is for Open AI not Applicable for Gemini

In [7]:
#7) Self-Ask with Search (clarifies, then searches)

search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Yes.
Follow up: When was the first LangChain release?
Intermediate answer: [{'title': 'LangChain - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/LangChain', 'content': '## History\n\n[edit]\n\nLangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark "Benchmark (venture capital firm)").\n\nIn the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.\n\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API. [...] | LangChain |\n\n| Developer | Harrison Chase |\n| Release | October 2022 |\n| Stable release | 0.1.16 / 1

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Follow up: Who founded LangChain?
Intermediate answer: [{'title': 'About LangChain: an open platform to own your intelligence', 'url': 'https://www.langchain.com/about', 'content': "Harrison teamed up with co-founder Ankush Gola to start LangChain, the company, in early 2023, and we've been building ahead of the industry ever since. After LangChain, we launched LangGraph to give developers control over complex agent systems, with a runtime built for production.\n\nAs we shipped more agents, we saw a gap in tooling for testing and debugging them. We launched LangSmith, an independent commercial platform, for AI observability. We quickly added services for agent evaluation, deployment, and a no-code agent builder. Most recently, we released an agent harness called Deep Agents for autonomous, long-horizon tasks. [...] We develop open source projects to stay ahead of the best patterns for building agents. Then, we incorporate these learnings into LangSmith - our commercial agent engineerin

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


So the final answer is: LangChain was first released in October 2022 and was founded by Harrison Chase (along with co-founder Ankush Gola).

> Finished chain.


Deep Drive

![Model Diagram](agenttypes1.png)

![Model Diagram](agenttype2.png)

![Model Diagram](pro1.png)

![Model Diagram](pro2.png)

![Model Diagram](pro3.png)